In [1]:
%pip install --upgrade pip
%pip install torch --index-url https://download.pytorch.org/whl/cpu
%pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_DIR = Path("smollm2_model")

print("Downloading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Downloading model...")

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

print("Saving model locally...")

tokenizer.save_pretrained(MODEL_DIR)
model.save_pretrained(MODEL_DIR, safe_serialization=True)

print("Model downloaded successfully!")
print("Model location:", MODEL_DIR.resolve())

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

C:\Users\Sakshi\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sakshi\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-360M-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

... (5 outputs left out)

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Saving model locally...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model downloaded successfully!
Model location: C:\Users\Sakshi\OneDrive\Desktop\DS\smollm2_model


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(4)

MODEL_DIR = "smollm2_model"

print("Loading local model...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model.to("cpu")
model.eval()

print("Model loaded successfully!")
print("Running on CPU.")

Loading local model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Running on CPU.


In [7]:
SYSTEM_PROMPT = """
You are a helpful AI teaching assistant.
Explain concepts in simple English.
Use examples whenever helpful.
If you are unsure about an answer, say so.
"""

MAX_NEW_TOKENS = 120
MAX_HISTORY_MESSAGES = 6


def generate_response(user_message, history=None):

    if history is None:
        history = []

    # Keep recent conversation messages
    recent_history = history[-MAX_HISTORY_MESSAGES:]

    # Prepare the conversation
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ] + recent_history + [
        {
            "role": "user",
            "content": user_message
        }
    ]

    # Convert conversation into model input
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Generate a response
    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    input_length = inputs["input_ids"].shape[1]

    new_tokens = output_ids[0][input_length:]

    # Convert tokens to readable text
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        answer = "I could not generate a response. Please try again."

    return answer


print("Chatbot function created successfully!")

Chatbot function created successfully!


In [8]:
question = "Explain Generative AI in simple terms."

answer = generate_response(question)

print("You:", question)
print("\nGenAI Chatbot:", answer)

You: Explain Generative AI in simple terms.

GenAI Chatbot: Generative AI is a type of artificial intelligence that creates new data or information based on patterns it has learned from past experiences. It can produce original content such as images, videos, music, text, etc., using algorithms and statistical models. This means that if you feed it with certain types of information, like faces, animals, or objects, it will generate something entirely different but related to those same things.

Think of it like being able to learn new words by looking at pictures of them and then being able to use those words to create sentences and other forms of language. Or imagine


In [9]:
chat_history = []

question = "What is Machine Learning?"

answer = generate_response(question, chat_history)

print("You:", question)
print("\nBot:", answer)

chat_history.extend([
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
])

You: What is Machine Learning?

Bot: Machine learning is a way of making computers learn and improve without being directly programmed to do so. It's like giving the computer special powers to think for itself and make decisions based on data it has been trained with.

Think of it like this: Imagine you have a big box of colored blocks and you want to teach your computer how to recognize different colors. You could give it lots of pictures of cars and houses and tell it which ones are blue, green, red, etc., but that would be very time-consuming. Instead, you use a machine learning algorithm (a set of instructions


In [10]:
question = "Give me a real-life example of it."

answer = generate_response(question, chat_history)

print("You:", question)
print("\nBot:", answer)

chat_history.extend([
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
])

You: Give me a real-life example of it.

Bot: A great real-life example is Netflix. They train their algorithms to recommend movies and TV shows by analyzing people's viewing habits and preferences. For instance, if someone watches a lot of sci-fi movies, they might get suggested to watch more sci-fi movies or even films from the same director. This way, Netflix can show personalized content to its users based on what they've liked before.


In [11]:
def start_chat():

    history = []

    print("=" * 50)
    print("       OFFLINE GENAI CHATBOT")
    print("=" * 50)
    print("Ask any question.")
    print("Type /reset to clear history.")
    print("Type /exit to stop.")
    print("=" * 50)

    while True:

        question = input("\nYou: ").strip()

        if not question:
            continue

        if question.lower() == "/exit":
            print("Bot: Goodbye!")
            break

        if question.lower() == "/reset":
            history.clear()
            print("Bot: Conversation history cleared.")
            continue

        print("Bot: Thinking...")

        answer = generate_response(question, history)

        print("\nBot:", answer)

        history.extend([
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ])

        # Keep the history bounded
        if len(history) > 12:
            history = history[-12:]


# Run the chatbot
start_chat()

       OFFLINE GENAI CHATBOT
Ask any question.
Type /reset to clear history.
Type /exit to stop.



You:  explain sql


Bot: Thinking...

Bot: SQL (Structured Query Language) is used to manage and store data in a database. It allows users to create tables with columns and rows, as well as manipulate the data by adding or removing records, updating existing records, and performing calculations on specific fields. SQL is often used for tasks such as:

1. Creating databases: You can use SQL commands to create new databases, insert data into them, and delete them.
2. Inserting data into tables: You can add data from various sources into your database, like selecting values from another table and inserting them into your current table.



You:  /exit


Bot: Goodbye!
